# 🎵 LSTM-Based Audio Sequence Prediction
### Lab Assignment 5 — Group Assignment

**Dataset:** Mozilla Common Voice (via Hugging Face)  
**Deployment:** FastAPI on Render.com

---
## Dataset Declaration
| Field | Details |
|---|---|
| **Name** | Mozilla Common Voice 11.0 (English) |
| **Source** | https://huggingface.co/datasets/mozilla-foundation/common_voice_11_0 |
| **Description** | Crowd-sourced speech recordings. We extract MFCC features from audio clips. |
| **Preprocessing** | Resample 16kHz → 13 MFCCs → Normalize → Sliding window sequences |

## Step 1: Install Dependencies

In [ ]:
!pip install librosa datasets soundfile tensorflow scikit-learn -q
print('✅ All packages installed')

## Step 2: Imports

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import os, json, pickle, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Step 3: Load Dataset

In [ ]:
from datasets import load_dataset

print('Loading Mozilla Common Voice...')
dataset = load_dataset(
    'mozilla-foundation/common_voice_11_0',
    'en',
    split='train[:500]',
    trust_remote_code=True
)
print(f'✅ Loaded {len(dataset)} audio samples')
print(f'Sample: {dataset[0]["sentence"]}')

## Step 4: Extract MFCC Features

In [ ]:
TARGET_SR  = 16000
N_MFCC     = 13
HOP_LENGTH = 512
N_FFT      = 2048
SEQ_LEN    = 20
PRED_LEN   = 5

def extract_mfcc(audio_array, sr):
    if sr != TARGET_SR:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=TARGET_SR)
    mfccs = librosa.feature.mfcc(
        y=audio_array.astype(np.float32),
        sr=TARGET_SR, n_mfcc=N_MFCC,
        hop_length=HOP_LENGTH, n_fft=N_FFT
    )
    return mfccs.T  # (T, n_mfcc)

print('Extracting MFCCs...')
all_mfccs = []
for i, sample in enumerate(dataset):
    audio = sample['audio']
    mfcc  = extract_mfcc(audio['array'], audio['sampling_rate'])
    if len(mfcc) > SEQ_LEN + PRED_LEN:
        all_mfccs.append(mfcc)
    if (i+1) % 100 == 0:
        print(f'  {i+1}/{len(dataset)} done')

print(f'✅ Valid samples: {len(all_mfccs)}')

## Step 5: Visualize MFCCs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
img = librosa.display.specshow(all_mfccs[0].T, x_axis='frames', ax=axes[0], cmap='coolwarm')
axes[0].set_title('MFCC Features — Sample 0')
fig.colorbar(img, ax=axes[0])
axes[1].plot(all_mfccs[0][:, 0], color='steelblue')
axes[1].set_title('MFCC[0] — Energy over time')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mfcc_visualization.png', dpi=150)
plt.show()

## Step 6: Normalize + Create Sequences

In [ ]:
# Normalize
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(np.vstack(all_mfccs))
normalized = [scaler.transform(m) for m in all_mfccs]

# Sliding window sequences
def create_sequences(mfcc_list):
    X, Y = [], []
    for mfcc in mfcc_list:
        for start in range(len(mfcc) - SEQ_LEN - PRED_LEN):
            X.append(mfcc[start : start + SEQ_LEN])
            Y.append(mfcc[start + SEQ_LEN : start + SEQ_LEN + PRED_LEN])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X, Y = create_sequences(normalized)
Y_flat = Y.reshape(Y.shape[0], -1)

X_temp, X_test, Y_temp, Y_test = train_test_split(X, Y_flat, test_size=0.1,  random_state=42)
X_train, X_val,  Y_train, Y_val  = train_test_split(X_temp, Y_temp, test_size=0.111, random_state=42)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## Step 7: Build LSTM Model

### Mathematical Model

**Forget gate:** $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$  
**Input gate:** $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$  
**Cell state:** $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$  
**Output gate:** $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$  
**Hidden state:** $h_t = o_t \odot \tanh(C_t)$

In [ ]:
model = Sequential([
    LSTM(128, input_shape=(SEQ_LEN, N_MFCC), return_sequences=True,
         dropout=0.2, recurrent_dropout=0.1, name='lstm_1'),
    BatchNormalization(),
    LSTM(64, return_sequences=False,
         dropout=0.2, recurrent_dropout=0.1, name='lstm_2'),
    BatchNormalization(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(PRED_LEN * N_MFCC, activation='linear', name='output')
], name='LSTM_AudioPredictor')

model.compile(optimizer=Adam(1e-3), loss='mse', metrics=['mae'])
model.summary()

## Step 8: Train

In [ ]:
history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=60, batch_size=64,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
    ],
    verbose=1
)

## Step 9: Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['mae'], label='Train')
axes[1].plot(history.history['val_mae'], label='Val')
axes[1].set_title('Mean Absolute Error')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## Step 10: Evaluate & Predict

In [ ]:
test_loss, test_mae = model.evaluate(X_test, Y_test, verbose=0)
print(f'Test MSE : {test_loss:.6f}')
print(f'Test MAE : {test_mae:.6f}')

# Single sample prediction
pred = model.predict(X_test[:1], verbose=0)
pred_frames   = scaler.inverse_transform(pred.reshape(PRED_LEN, N_MFCC))
target_frames = scaler.inverse_transform(Y_test[:1].reshape(PRED_LEN, N_MFCC))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(target_frames.T, aspect='auto', cmap='coolwarm')
axes[0].set_title('Ground Truth (next 5 frames)')
axes[1].imshow(pred_frames.T, aspect='auto', cmap='coolwarm')
axes[1].set_title('LSTM Prediction (next 5 frames)')
plt.tight_layout()
plt.savefig('prediction_vs_actual.png', dpi=150)
plt.show()

## Step 11: Save Model Files

In [ ]:
os.makedirs('saved_model', exist_ok=True)

model.save('saved_model/lstm_audio_model.keras')
with open('saved_model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('saved_model/config.json', 'w') as f:
    json.dump({
        'target_sr': TARGET_SR, 'n_mfcc': N_MFCC,
        'hop_length': HOP_LENGTH, 'n_fft': N_FFT,
        'seq_len': SEQ_LEN, 'pred_len': PRED_LEN
    }, f, indent=2)

print('✅ Model saved to saved_model/')
!ls -lh saved_model/

## Step 12: Push Everything to GitHub
**Run this cell to upload model files + API code to your GitHub repo.**

Fill in your GitHub username, repo name, and token before running.

In [ ]:
# ──────────────────────────────────────────────
# FILL THESE IN BEFORE RUNNING
GITHUB_USERNAME = "your_username"       # e.g. "rahul123"
GITHUB_REPO     = "lstm-audio-prediction"  # your repo name
GITHUB_TOKEN    = "ghp_xxxxxxxxxxxx"    # your GitHub personal access token
# ──────────────────────────────────────────────

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"

!git config --global user.email "you@example.com"
!git config --global user.name "{GITHUB_USERNAME}"

# Clone the repo
!git clone {REPO_URL} /content/myrepo

# Copy all files into it
import shutil
shutil.copytree('saved_model', '/content/myrepo/saved_model', dirs_exist_ok=True)

# Copy plots
for f in ['mfcc_visualization.png', 'training_curves.png', 'prediction_vs_actual.png']:
    if os.path.exists(f):
        shutil.copy(f, f'/content/myrepo/{f}')

# Commit and push
import subprocess
subprocess.run(['git', '-C', '/content/myrepo', 'add', '.'], check=True)
subprocess.run(['git', '-C', '/content/myrepo', 'commit', '-m', 'Add trained LSTM model and plots'], check=True)
subprocess.run(['git', '-C', '/content/myrepo', 'push'], check=True)

print('✅ All files pushed to GitHub!')
print(f'🔗 https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}')